# Spectral Diagnostic for LoRA Rank Utilization: Qwen

This notebook runs the empirical spectral diagnostic and SEE-Rank evaluation for **Qwen/Qwen2.5-1.5B**.

### Experimental Pipeline:
1. **Rank sweep**: Evaluates ranks $r \in \{4, 8, 16, 32, 64\}$ with $\alpha = 2r$.
2. **Regularization conditions**: Evaluates weight decay ($\lambda = 0.10$), dropout ($p = 0.20$), and DoRA at $r = 16$.
3. **Spectral checkpointing**: Computes exact QR-based SVD across adapter layers at fine-grained early steps.
4. **Evaluation**: Validation perplexity, TruthfulQA MC1, ARC-Easy, and SVD truncation construct validity.
5. **SEE-Rank**: Probe-and-stop decision rule across $\tau \in \{0.10, 0.15, 0.20, 0.25, 0.30\}$.

In [ ]:
# Environment setup and dependencies
!pip install -q "bitsandbytes>=0.46.1" "peft>=0.14" datasets accelerate scipy tqdm pandas


In [ ]:
import os, sys
import torch

# Add parent directory to path to import src
sys.path.append(os.path.abspath(".."))

print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
else:
    print("Running on CPU")


In [ ]:
from run_experiment import main as execute_experiment

# Configure execution for this model
import sys
sys.argv = ["run_experiment.py", "--model", "Qwen/Qwen2.5-1.5B", "--out-dir", "../results/qwen"]

# Execute full diagnostic sweep
execute_experiment()


In [ ]:
import json
import pandas as pd

out_dir = "../results/qwen"
model_short = "Qwen2.5-1.5B"
results_path = f"{out_dir}/spectral_{model_short}_results.csv"
seerank_path = f"{out_dir}/spectral_{model_short}_see_rank.csv"
forms_path = f"{out_dir}/spectral_{model_short}_functional_forms.json"

if os.path.exists(results_path):
    df_res = pd.read_csv(results_path)
    print("=== Summary Results Table ===")
    print(df_res[["method", "rank", "seed", "final_ppl", "mean_sr_final", "truthfulqa_mc1", "arc_easy"]])

if os.path.exists(seerank_path):
    df_see = pd.read_csv(seerank_path)
    print("\n=== SEE-Rank Evaluation ===")
    print(df_see)

if os.path.exists(forms_path):
    with open(forms_path) as f:
        forms = json.load(f)
    print("\n=== Functional Form Comparison (AIC) ===")
    for k, v in forms.items():
        print(f"{k}: {v}")
